# Volume 4, Chapter 2: Generating Functions as a Computing Tool (Python companion)

Exercises first, solutions at the end. We turn generating functions into algorithms: the coin-change DP, exact costs of merge sort and quicksort against measurements, two ways to compute partition numbers, the logarithm of a power series to count connected graphs, and fast exact Fibonacci numbers against the floating-point Binet formula.

*Before each solution, predict what the output will look like.*

## Exercises

**P2.1 (★) Making change.** Compute the number of ways to pay $1$ euro (Exercise 2.2) with the running-sum algorithm, and check it against the coefficient of $x^{100}$ computed by SymPy. Then count the ways to pay $10$ euros with all coins and the $5$ and $10$ euro notes.

**P2.2 (★★) CS application: exact costs of sorting.** Count the comparisons of merge sort on random inputs of size $n=2^m$ and compare with the bound $n\log_2n$ of Example 2.7. Average the comparisons of randomised quicksort over $200$ runs and compare with $2(n+1)H_n-4n$ (Example 2.8). Which algorithm wins, and by how much?

**P2.3 (★★) Partition numbers, two ways.** Compute $p(n)$ for $n\le3000$ by the product $\prod\frac1{1-x^k}$ (a DP in $O(n^2)$ additions) and by Euler's recurrence (Corollary 2.13), check MacMahon's $p(200)=3\,972\,999\,029\,388$, and time both.

**P2.4 (★★) The logarithm of a series counts connected graphs.** Compute the logarithm of $\sum2^{\binom n2}\frac{x^n}{n!}$ exactly up to $n=12$ (Exercise 2.5), check $c_5=728$ by brute force over the $2^{10}$ graphs on $5$ vertices, and watch the proportion of connected graphs.

**P2.5 (★) Exact against floating point.** Compute $F_n$ by the doubling formulas $F_{2k}=F_k(2F_{k+1}-F_k)$, $F_{2k+1}=F_k^2+F_{k+1}^2$, which need $O(\log n)$ multiplications, and compare with Binet's formula evaluated in floating point. Where does the float version first fail? Then ask SymPy to solve a recurrence whose characteristic polynomial has a double root (Theorem 2.5).

**P2.6 (★★★) Euler's divisor recurrence.** Check Exercise 2.7 for all $n\le1000$.

## Solutions

### P2.1

In [ ]:
import sympy as sp
x = sp.symbols('x')
coins = [1, 2, 5, 10, 20, 50, 100]
def ways(total, coins):
    b = [1] + [0] * total
    for c in coins:                      # multiply by 1/(1 - x^c): a running sum with step c
        for n in range(c, total + 1):
            b[n] += b[n - c]
    return b[total]
print('ways to pay 1 euro:', ways(100, coins))
prod = sp.prod([sum(x**(c * k) for k in range(100 // c + 1)) for c in coins])
print('coefficient of x^100 from SymPy:', sp.Poly(sp.expand(prod), x).coeff_monomial(x**100))
print('ways to pay 10 euros (euro coins, 2 euro coins, 5 and 10 euro notes):', ways(1000, coins + [200, 500, 1000]))

Both give $4563$. The running-sum algorithm applies the seven factors in about $700$ additions, while the polynomial product computes all coefficients up to degree $100$ of each factor: same answer, much more work. With $10$ euros and notes the count is already $327\,631\,322$, still instant for the DP.

### P2.2

In [ ]:
import random, math
random.seed(2)
def merge_sort(a, counter):
    if len(a) <= 1:
        return a
    m = len(a) // 2
    left, right = merge_sort(a[:m], counter), merge_sort(a[m:], counter)
    out, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        counter[0] += 1
        if left[i] <= right[j]: out.append(left[i]); i += 1
        else: out.append(right[j]); j += 1
    return out + left[i:] + right[j:]
def quick_sort(a, counter):
    if len(a) <= 1:
        return a
    p = random.choice(a)
    counter[0] += len(a) - 1
    return quick_sort([v for v in a if v < p], counter) + [p] + quick_sort([v for v in a if v > p], counter)
print('    n   merge sort (random input)   n log2 n    quicksort (mean of 200)   2(n+1)H_n - 4n')
for m in (4, 7, 10, 13):
    n = 2**m
    c = [0]; merge_sort(random.sample(range(n), n), c)
    q = 0
    for _ in range(200):
        cq = [0]; quick_sort(random.sample(range(n), n), cq); q += cq[0]
    H = sum(1 / k for k in range(1, n + 1))
    print(f'{n:5d}   {c[0]:12d}               {n * m:8d}    {q / 200:12.1f}            {2 * (n + 1) * H - 4 * n:10.1f}')

Merge sort stays below its worst case $n\log_2n$; on random inputs it uses about $n\log_2n-1.25n$ comparisons (Knuth, Section 5.2.4), since merges usually stop before both halves are exhausted. The averages of quicksort match $2(n+1)H_n-4n$ to within about $1\%$, the size of the fluctuations over $200$ runs. Asymptotically $2n\ln n=1.39\,n\log_2n$: quicksort makes about $39\%$ more comparisons than merge sort, and is still often faster in practice, because its inner loop is simpler and it sorts in place.

### P2.3

In [ ]:
import time
def partitions_dp(N):
    p = [1] + [0] * N
    for k in range(1, N + 1):
        for n in range(k, N + 1):
            p[n] += p[n - k]
    return p
def partitions_pentagonal(N):
    p = [1] + [0] * N
    for n in range(1, N + 1):
        s, j = 0, 1
        while True:
            g1, g2 = j * (3 * j - 1) // 2, j * (3 * j + 1) // 2
            if g1 > n: break
            sign = 1 if j % 2 else -1
            s += sign * p[n - g1]
            if g2 <= n: s += sign * p[n - g2]
            j += 1
        p[n] = s
    return p
for f in (partitions_dp, partitions_pentagonal):
    t = time.perf_counter(); P = f(3000); t = time.perf_counter() - t
    print(f'{f.__name__:22s} p(200) = {P[200]},  p(1000) = {P[1000]},  time for n <= 3000: {t:.2f} s')
assert partitions_dp(500) == partitions_pentagonal(500)

Both methods agree, and MacMahon's value is right. Euler's recurrence uses about $\sqrt{8n/3}$ terms per value instead of $n$, which here makes it more than ten times faster; $p(1000)$ already has $32$ digits, a growth like $e^{\pi\sqrt{2n/3}}$ that Chapter 11 explains.

### P2.4

In [ ]:
from fractions import Fraction
from math import comb, factorial
N = 12
F = [Fraction(2**comb(n, 2), factorial(n)) for n in range(N + 1)]   # all graphs, EGF coefficients
def log_series(F, N):
    """log(F) for F[0] = 1, from F * (log F)' = F'."""
    L = [Fraction(0)] * (N + 1)
    for n in range(1, N + 1):
        L[n] = F[n] - sum(Fraction(k, n) * L[k] * F[n - k] for k in range(1, n))
    return L
L = log_series(F, N)
c = [int(L[n] * factorial(n)) for n in range(N + 1)]
print('connected graphs c_1..c_8:', c[1:9])
print('fraction of graphs that are connected:', [round(c[n] / 2**comb(n, 2), 4) for n in range(1, N + 1)])
import itertools
def connected(n, edges):
    parent = list(range(n))
    def find(a):
        while parent[a] != a: a = parent[a]
        return a
    for a, b in edges: parent[find(a)] = find(b)
    return len({find(v) for v in range(n)}) == 1
pairs = list(itertools.combinations(range(5), 2))
brute = sum(connected(5, [e for e, keep in zip(pairs, mask) if keep]) for mask in itertools.product([0, 1], repeat=len(pairs)))
print('brute force on 5 vertices:', brute)

The logarithm, computed from $F\,(\log F)'=F'$ with exact fractions, gives $1,1,4,38,728,26704,\dots$, and brute force confirms $728$. The proportion of connected graphs dips to $\frac12$ at $n=2,3$ and then tends to $1$: a uniformly random graph has edge probability $\frac12$, far above the connectivity threshold $\frac{\ln n}n$ of Volume 3, Chapter 12.

### P2.5

In [ ]:
def fib_doubling(n):
    """(F_n, F_{n+1}) with O(log n) multiplications: F_2k = F_k(2F_{k+1}-F_k), F_2k+1 = F_k^2+F_{k+1}^2."""
    if n == 0: return 0, 1
    a, b = fib_doubling(n // 2)
    c, d = a * (2 * b - a), a * a + b * b
    return (d, c + d) if n % 2 else (c, d)
phi = (1 + 5 ** 0.5) / 2
first_bad = next(n for n in range(200) if fib_doubling(n)[0] != round(phi**n / 5**0.5))
print('first n where the floating-point Binet formula is wrong:', first_bad)
for n in (10, 70, 71, 72, 80):
    exact = fib_doubling(n)[0]
    print(f'n = {n}: exact F_n = {exact},  round(phi^n/sqrt5) = {round(phi**n / 5**0.5)},  agree: {exact == round(phi**n / 5**0.5)}')
t = time.perf_counter(); F6 = fib_doubling(10**6)[0]; t = time.perf_counter() - t
import sys; sys.set_int_max_str_digits(0)
print(f'F_(10^6) has {len(str(F6))} digits, computed in {t:.2f} s')
u = sp.Function('u'); n = sp.symbols('n', integer=True)
print('closed form of u_(n+3) = 3u_(n+2) - 4u_n, u = 1, 0, 1:', sp.rsolve(u(n + 3) - 3 * u(n + 2) + 4 * u(n), u(n), {u(0): 1, u(1): 0, u(2): 1}))

In double precision Binet's formula is exact for $n\le70$ and first fails at $n=71$: $F_{71}\approx3\times10^{14}$ is still far below $2^{53}\approx9\times10^{15}$, but the rounding error of $\varphi$ itself, about $10^{-16}$ in relative terms, is multiplied by $n$ in $\varphi^n$, and $71\times10^{-16}\times3\times10^{14}$ is already more than $\frac12$. The doubling formulas compute the $208\,988$ digits of $F_{10^6}$ exactly in a fraction of a second, because Python multiplies big integers with Karatsuba's algorithm; Chapter 7 does better with the FFT. SymPy's answer for the recurrence with characteristic polynomial $y^3-3y^2+4=(y-2)^2(y+1)$ has the form $(a+bn)2^n+c(-1)^n$, as Theorem 2.5 predicts.

### P2.6

In [ ]:
def sigma(m): return sum(d for d in range(1, m + 1) if m % d == 0)
S = [0] + [sigma(m) for m in range(1, 1001)]
def euler_sigma(n):
    s, j = 0, 1
    while True:
        for g in (j * (3 * j - 1) // 2, j * (3 * j + 1) // 2):
            if g <= n:
                s += (1 if j % 2 else -1) * (S[n - g] if g < n else n)
        if j * (3 * j - 1) // 2 > n: break
        j += 1
    return s
print('Euler\'s recurrence for sigma holds for n <= 1000:', all(euler_sigma(m) == S[m] for m in range(1, 1001)))

The recurrence holds for every $n\le1000$. Euler had no proof of the pentagonal theorem when he found this law in 1747; he proved it in 1750, and Franklin's bijection came $131$ years later.